# Building plugin caches from source — validation run for datafusion-bio-functions #217

Rebuilds every vepyr plugin cache **from its raw source**, one plugin at a time, against
the merged [datafusion-bio-functions#217](https://github.com/biodatageeks/datafusion-bio-functions/pull/217) plus its native indexed-TSV follow-up.

This is the real-chromosome spot-check both reviews on #217 asked for and that the
unit tests cannot stand in for. It exercises, on real data:

| #217 change | Exercised by |
|---|---|
| `ProviderKind::Vcf` wiring | **clinvar**, **spliceai** (`provider = "vcf"`) |
| Final spill-capable `ORDER BY tier, start` after adaptive joins | **clinvar** — sparse enough to exercise a different join shape |
| `assume_unique` + `check_assume_unique_sample` | **cadd**, **spliceai** (`assume_unique = true`) |
| Streaming shard write (the OOM case) | **cadd** chr1-2, **dbnsfp** |
| Keep-first dedup (must NOT be skipped) | **alphamissense** (overlapping UniProts) |
| `ScratchGuard` cleanup | any interrupted build (see the disk-hygiene check) |
| `ProviderKind::Bed` | **not covered** — no BED manifest exists in vepyr-plugins |

## Run order is deliberate

Smallest and newest-code-path first, so a broken pipeline is found in minutes rather
than after an 87 GB download:

1. **clinvar** — 192 MB, VCF provider, sparse adaptive-join path
2. **alphamissense** — 628 MB, dedup path, has a known-good prior result to diff against
3. **spliceai** — 28.5 GB, VCF provider + `assume_unique`
4. **dbnsfp** — 50.3 GB
5. **cadd** — 87.5 GB, `assume_unique` + SNV/indel merge, the OOM case #196 was built for

## Disk is the binding constraint

Sources total **~168 GB compressed** against **~234 GB free**. CADD alone is 87.5 GB
compressed. Rust queries one chromosome from each BGZF/tabix source and stages only that
region uncompressed for parallel parsing, but the source plus build scratch still needs headroom. Hence: **download one plugin, build it,
delete its input, keep only the parquet cache.** Never hold two large sources at once.
Every section below ends with a cleanup cell — do not skip it.

## 0. Setup

Paths, remote, and the per-plugin source matrix.

In [ ]:
import shutil
import subprocess
import sys
import time
from pathlib import Path

DATA = Path("/Users/mwiewior/workspace/data_vepyr")
INPUT_ROOT = DATA / "plugin_input"  # scratch: deleted per plugin after its build
CACHE_ROOT = DATA / "plugin_cache"  # kept: the actual deliverable
VARIATION = DATA / "cache" / "116_GRCh38_merged"  # tiering is inherited from this
PLUGINS_REPO = Path("/tmp/vepyr-plugins")  # checkout of the manifests repo
PLUGINS_REF = "8af8353899ea44b36c80783b781f8bfb5913b242"  # vepyr-plugins master: PRs #2, #6, #7 merged

DRIVE_REMOTE = "gdrive-mw"
DRIVE_FOLDER = "1ZT3g31I0LXepORF_dusy47AuXOnbRlZ_"

INPUT_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
assert VARIATION.is_dir(), f"variation cache missing: {VARIATION}"
print(
    "vepyr:",
    subprocess.run(
        [sys.executable, "-c", "import vepyr;print(vepyr.__file__)"],
        capture_output=True,
        text=True,
    ).stdout.strip(),
)

In [ ]:
# Fail now, not 80 GB into a download. Transfer and temporary VCF slicing are
# external; indexed TSV chromosome selection runs natively in Rust.
missing = [t for t in ("rclone", "tabix", "bgzip") if not shutil.which(t)]
assert not missing, f"missing tools: {missing} (htslib: `brew install htslib`)"
print("tools ok")

In [ ]:
# The manifests come from vepyr-plugins, resolved by `git worktree add <ref>` -- so the
# ref must be COMMITTED in this checkout. Editing a .source.toml without committing
# silently rebuilds from the old manifest.
if not PLUGINS_REPO.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/biodatageeks/vepyr-plugins.git",
            str(PLUGINS_REPO),
        ],
        check=True,
    )
subprocess.run(
    ["git", "-C", str(PLUGINS_REPO), "fetch", "origin", PLUGINS_REF], check=True
)
subprocess.run(["git", "-C", str(PLUGINS_REPO), "checkout", PLUGINS_REF], check=True)
head = subprocess.run(
    ["git", "-C", str(PLUGINS_REPO), "rev-parse", "--short", "HEAD"],
    capture_output=True,
    text=True,
).stdout.strip()
print(f"manifests: {PLUGINS_REPO} @ {PLUGINS_REF} ({head})")
print(sorted(x.name for x in (PLUGINS_REPO / "plugins").iterdir()))

In [ ]:
# Per-plugin source matrix. `prefix` is needed only for notebook-sliced VCFs;
# Rust resolves indexed TSV contig aliases directly from each tabix index.
PLUGINS = {
    "clinvar": dict(
        files=["clinvar/clinvar.vcf.gz", "clinvar/clinvar.vcf.gz.tbi"],
        main="clinvar.vcf.gz",
        kind="vcf",
        prefix="",
        slice_ext=".vcf.gz",
        size_gb=0.19,
        note="sparse adaptive join; final DataFusion ORDER BY owns row order",
    ),
    "alphamissense": dict(
        files=[
            "alphamissense/AlphaMissense_hg38.bgz.tsv.gz",
            "alphamissense/AlphaMissense_hg38.bgz.tsv.gz.tbi",
        ],
        main="AlphaMissense_hg38.bgz.tsv.gz",
        kind="indexed_tsv",
        prefix="chr",
        slice_ext=".tsv.gz",
        size_gb=0.63,
        note="Rust queries BGZF/tabix per chrom; dedup MUST run (overlapping UniProts)",
    ),
    "spliceai": dict(
        files=[
            "spliceai/spliceai_scores.masked.snv.ensembl_mane.grch38.110.vcf.gz",
            "spliceai/spliceai_scores.masked.snv.ensembl_mane.grch38.110.vcf.gz.tbi",
        ],
        main="spliceai_scores.masked.snv.ensembl_mane.grch38.110.vcf.gz",
        kind="vcf",
        prefix="",
        slice_ext=".vcf.gz",
        size_gb=28.5,
        note="assume_unique=true -> check_assume_unique_sample runs",
    ),
    "dbnsfp": dict(
        files=["dbnsfp/dbNSFP5.3.1a_grch38.gz", "dbnsfp/dbNSFP5.3.1a_grch38.gz.tbi"],
        main="dbNSFP5.3.1a_grch38.gz",
        kind="indexed_tsv",
        prefix="",
        slice_ext=".tsv.gz",
        size_gb=50.3,
        note="Rust queries the canonical BGZF/tabix source per chromosome",
    ),
    "cadd": dict(
        files=[
            "cadd/whole_genome_SNVs.tsv.gz",
            "cadd/whole_genome_SNVs.tsv.gz.tbi",
            "cadd/gnomad.genomes.r4.0.indel.tsv.gz",
            "cadd/gnomad.genomes.r4.0.indel.tsv.gz.tbi",
        ],
        # Two [[source]] parts, combined by UNION ALL in the manifest's ingest_sql.
        parts={
            "snv": "whole_genome_SNVs.tsv.gz",
            "indel": "gnomad.genomes.r4.0.indel.tsv.gz",
        },
        kind="indexed_tsv",
        prefix="",
        slice_ext=".tsv.gz",
        size_gb=88.8,
        note="Rust queries both BGZF/tabix parts per chrom; SQL UNION ALL combines them",
    ),
}
for n, p in PLUGINS.items():
    print(
        f"{n:<14} {p['size_gb']:>6.1f} GB  {p['kind']:<7} prefix={p['prefix']!r:<6} {p['note']}"
    )

In [ ]:
def sh(cmd, **kw):
    """Run a shell command, streaming failure output rather than swallowing it.

    Runs under bash with `pipefail`, so a failing producer in a pipeline is not
    hidden by a successful consumer: `tabix ... | bgzip -c > out` otherwise
    reports bgzip's status and happily writes a header-only slice when tabix
    found nothing. /bin/sh is not enough -- it is dash on many Linux distros,
    which has no `pipefail`.
    """
    print("$", cmd)
    r = subprocess.run(
        f"set -o pipefail; {cmd}",
        shell=True,
        executable="/bin/bash",
        text=True,
        capture_output=True,
        **kw,
    )
    if r.returncode != 0:
        print(r.stdout[-4000:])
        print(r.stderr[-4000:])
        raise RuntimeError(f"rc={r.returncode}: {cmd}")
    return r.stdout


def free_gb(path=DATA):
    return shutil.disk_usage(path).free / 2**30


def require_space(gb, why=""):
    have = free_gb()
    print(f"free: {have:.1f} GB, need ~{gb:.1f} GB  {why}")
    if have < gb:
        raise RuntimeError(
            f"insufficient disk: {have:.1f} GB free, need {gb:.1f} GB. "
            "Delete a previous plugin's input first."
        )


def download(plugin):
    """Fetch only this plugin's files. Sources are pulled one plugin at a time by design."""
    spec = PLUGINS[plugin]
    dest = INPUT_ROOT / plugin
    dest.mkdir(parents=True, exist_ok=True)
    # 2.5x headroom = the compressed source (1x) plus one Rust-managed plain
    # chrom stage (1.5x). Bytes already on disk must not be charged twice: on
    # resume the source is present and rclone reuses it, so demanding another
    # full copy aborts a build that would have run. CADD is 88.8 GB, and its
    # documented host has ~145 GB left once it has landed.
    have_gb = (
        sum(
            (dest / Path(rel).name).stat().st_size
            for rel in spec["files"]
            if (dest / Path(rel).name).exists()
        )
        / 2**30
    )
    to_fetch_gb = max(0.0, spec["size_gb"] - have_gb)
    require_space(
        to_fetch_gb + spec["size_gb"] * 1.5,
        f"({plugin}: {to_fetch_gb:.1f} GB to fetch + chrom scratch; "
        f"{have_gb:.1f} GB already present)",
    )
    for rel in spec["files"]:
        sh(
            f'rclone copy "{DRIVE_REMOTE}:{rel}" "{dest}/" '
            f"--drive-root-folder-id {DRIVE_FOLDER} --low-level-retries 50 --progress --stats 30s"
        )
    for rel in spec["files"]:  # rclone can exit 0 having copied nothing
        f = dest / Path(rel).name
        assert f.exists() and f.stat().st_size > 0, f"download silently failed: {f}"
        print(f"  ok {f.name}  {f.stat().st_size / 2**30:.2f} GB")

## 1. Slice + build helpers

One chromosome per `build_plugin_cache` call — never the whole genome at once. Memory is
bounded (~3-4 GB RSS) by #217's streaming write regardless of chromosome size; **time**,
not memory, is the binding constraint.

In [ ]:
def source_for_chrom(plugin, chrom):
    """Return the source path(s) plus any notebook-owned temporary slices.

    Indexed TSV sources stay whole: the Rust builder uses BGZF/tabix to query
    only this chromosome. Indexed VCF manifests declare the same source-level
    contract, but retain a small notebook-owned slice until the builder passes an
    explicit region to their provider. No sorting or reshaping occurs.
    """
    spec = PLUGINS[plugin]
    d = INPUT_ROOT / plugin
    region = f"{spec['prefix']}{chrom}"
    if spec["kind"] == "indexed_tsv":
        if "parts" in spec:
            return ({part: d / fname for part, fname in spec["parts"].items()}, [])
        return (d / spec["main"], [])

    def has_records(path):
        """True when the slice holds at least one non-header line.

        Size proves nothing: -h emits the header, so a region matching no
        records still yields a non-empty file -- and validate() accepts a
        zero-row shard, because ordering and uniqueness hold vacuously over no
        rows. That would publish a chromosome with every plugin field absent.
        """
        return bool(sh(f'bgzip -dc "{path}" | grep -v "^#" | head -1 || true').strip())

    def one(src, out, kind):
        if out.exists():
            # A slice from an earlier run is not automatically trustworthy: a
            # previous attempt may have written a header-only file. Reuse has
            # to clear the same bar as creation, or the check is one `exists()`
            # away from being skipped entirely.
            if has_records(out):
                print(f"  reusing {out.name}")
                return out
            print(f"  discarding header-only {out.name} left by an earlier run")
            out.unlink()

        assert kind == "vcf", f"unexpected notebook-sliced kind: {kind}"
        # -h keeps the header: the VCF provider needs it to type INFO columns.
        # Written under a partial name and published only once it has records,
        # so a failure cannot leave an invalid slice for the next run to find.
        partial = out.with_name(out.name + ".partial")
        sh(f'tabix -h "{src}" {region} | bgzip -c > "{partial}"')
        if not has_records(partial):
            partial.unlink()
            raise AssertionError(
                f"slice for {plugin} {region} has a header but no records -- "
                f"wrong contig prefix, or the region is absent from {src.name}"
            )
        partial.rename(out)
        print(f"  {out.name}  {out.stat().st_size / 2**30:.2f} GB")
        return out

    sliced = one(
        d / spec["main"], d / f"{plugin}_chr{chrom}{spec['slice_ext']}", spec["kind"]
    )
    return sliced, [sliced]

In [ ]:
import vepyr

# Build inputs that are derived artifacts of the manifest's `url` can never
# hash to its md5: AlphaMissense's is a BGZF re-compression of the upstream
# plain gzip (see plugins/alphamissense/README.md in vepyr-plugins). "warn"
# records the digest actually found instead of refusing.
VERIFY_SOURCE = {"alphamissense": "warn"}


def build_chrom(plugin, chrom, keep_slice=False):
    """Select/query one chromosome, build it, and drop notebook-owned slices."""
    sl, temporary = source_for_chrom(plugin, chrom)
    source_path = (
        {k: str(v) for k, v in sl.items()} if isinstance(sl, dict) else str(sl)
    )
    t0 = time.time()
    result = vepyr.build_plugin_cache(
        plugin,
        PLUGINS_REF,
        source_path=source_path,
        cache_dir=str(VARIATION),
        plugin_cache_root=str(CACHE_ROOT),
        chroms=[str(chrom)],
        plugins_repo=str(PLUGINS_REPO),  # resolves the ref via `git worktree add`
        overwrite=True,
        # A per-contig slice cut here can never hash to the whole file's md5,
        # so the source check is skipped for slices; a source used whole is
        # verified strictly.
        verify_source="skip" if temporary else VERIFY_SOURCE.get(plugin, True),
    )
    el = time.time() - t0
    for c, rows, warm, cold in result:
        print(
            f"  {plugin} chr{c}: rows={rows:,} warm={warm:,} cold={cold:,}  [{el / 60:.1f} min]"
        )
    if not keep_slice:
        for f in temporary:
            f.unlink(missing_ok=True)
    return result

## 2. Validation

`rc=0` is not proof of a correct build. Two invariants are checked on every shard, both
of which #217 touches directly:

- **`(tier, start)` ascending** — `PageDir::resolve_ranges` binary-searches within a tier,
  so an unsorted run silently misses rows that are present. The final spill-capable
  `ORDER BY tier, start` establishes the order and `assert_start_monotonic` verifies it
  at the storage boundary.
- **no duplicate probe keys** — for an `assume_unique` plugin a duplicate means the flag is
  wrong and the runtime `HashMap` would keep the *last* row, inverting VEP's first-in-file rule.
  This check is exhaustive, unlike the builder's 2 M-key sample.

In [ ]:
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

MATCH_COLS = {
    "alphamissense": ["protein_variant"],
    "spliceai": ["symbol"],
    "dbnsfp": ["aa_change"],
    "cadd": [],
    "clinvar": [],
}
ASSUME_UNIQUE = {"cadd", "spliceai"}

# CADD chr1 alone is ~700M rows, so nothing here may hold a row per record.
#
# Duplicate detection has to cover the whole shard, not one block of it. A
# shard is a warm block then a cold block, each independently start-sorted,
# and the SAME start legitimately appears in both -- 1,117,083 cold rows of
# CADD chr22 share a start with a warm row. So a duplicate probe key falls into
# exactly one of three cases:
#
#   warm x warm   caught by the warm key set, which is small enough to hold
#   cold x cold   same start, so contiguous inside the start-sorted cold block
#   warm x cold   only possible where a cold start also occurs in warm, which
#                 is a tiny filtered subset -- checked against the warm set
#
# Segmenting on (tier, start) alone would silently skip the third case.
BATCH_ROWS = 1 << 17


def _keys(table, key_cols):
    cols = [table.column(c).to_pylist() for c in key_cols]
    return list(zip(*cols)) if cols else []


def _trailing_run(table, col):
    """Split off the rows sharing the final value of `col`; they may continue."""
    n = table.num_rows
    if n == 0:
        return table, table
    last = table.column(col)[n - 1].as_py()
    run_len = (
        pc.sum(pc.cast(pc.equal(table.column(col), last), pa.int64())).as_py() or 0
    )
    return table.slice(0, n - run_len), table.slice(n - run_len)


def _dupes_within(table, key_cols):
    if table.num_rows == 0:
        return 0
    grouped = table.select(key_cols).group_by(key_cols).aggregate([([], "count_all")])
    return table.num_rows - grouped.num_rows


def validate(plugin, chrom):
    shard = CACHE_ROOT / "plugin" / plugin / f"chr{chrom}.parquet"
    assert shard.exists(), f"no shard written: {shard}"
    pf = pq.ParquetFile(shard)
    print(
        f"  {shard.name}: {pf.metadata.num_rows:,} rows, "
        f"{pf.metadata.num_columns} cols, "
        f"{shard.stat().st_size / 2**20:.0f} MB"
    )

    key_cols = (
        ["start", "allele_string", *MATCH_COLS[plugin]]
        if plugin in ASSUME_UNIQUE
        else []
    )
    read_cols = ["tier", "start"] + [c for c in key_cols if c not in ("tier", "start")]

    prev_tier = prev_start = None
    regressions = 0
    first_regression = None
    row_base = 0

    warm_keys = set()
    warm_starts = set()
    warm_start_values = None  # pa.array, built once the warm block ends
    pending = None
    n_dupes = n_keys = 0

    for batch in pf.iter_batches(batch_size=BATCH_ROWS, columns=read_cols):
        table = pa.Table.from_batches([batch])
        tier = table.column("tier")
        start = table.column("start")

        # Tiers grouped warm-then-cold; start non-decreasing inside a tier.
        # Checked against the previous row too, so a boundary violation is not
        # missed between batches.
        if prev_tier is not None:
            assert tier[0].as_py() >= prev_tier, (
                f"{plugin} chr{chrom}: tiers not grouped warm-then-cold "
                f"(row {row_base})"
            )
            if tier[0].as_py() == prev_tier and start[0].as_py() < prev_start:
                regressions += 1
                first_regression = first_regression or row_base
        if table.num_rows > 1:
            prev_t = tier.slice(0, table.num_rows - 1)
            next_t = tier.slice(1)
            assert not pc.any(pc.less(next_t, prev_t)).as_py(), (
                f"{plugin} chr{chrom}: tiers not grouped warm-then-cold "
                f"(within batch at row {row_base})"
            )
            regressed = pc.and_(
                pc.equal(next_t, prev_t),
                pc.less(start.slice(1), start.slice(0, table.num_rows - 1)),
            )
            n_bad = pc.sum(pc.cast(regressed, pa.int64())).as_py() or 0
            if n_bad and first_regression is None:
                first_regression = row_base + pc.index(regressed, True).as_py() + 1
            regressions += n_bad

        prev_tier = tier[table.num_rows - 1].as_py()
        prev_start = start[table.num_rows - 1].as_py()
        row_base += table.num_rows

        if not key_cols:
            continue

        warm = table.filter(pc.equal(tier, 0))
        cold = table.filter(pc.not_equal(tier, 0))

        if warm.num_rows:
            for k in _keys(warm, key_cols):
                if k in warm_keys:
                    n_dupes += 1
                else:
                    warm_keys.add(k)
            warm_starts.update(warm.column("start").to_pylist())
            n_keys += warm.num_rows

        if cold.num_rows:
            if pending is not None and pending.num_rows:
                cold = pa.concat_tables([pending, cold])
            complete, pending = _trailing_run(cold, "start")
            n_dupes += _dupes_within(complete, key_cols)
            n_keys += complete.num_rows
            if warm_keys:
                # Only cold rows at a start that also occurs in warm can
                # collide across the tier boundary; that is a small subset.
                # Warm always precedes cold, so the value set is final by the
                # time any cold row is seen -- build it once, not per batch.
                if warm_start_values is None:
                    warm_start_values = pa.array(sorted(warm_starts))
                overlap = complete.filter(
                    pc.is_in(complete.column("start"), value_set=warm_start_values)
                )
                for k in _keys(overlap, key_cols):
                    if k in warm_keys:
                        n_dupes += 1

    if key_cols and pending is not None and pending.num_rows:
        n_dupes += _dupes_within(pending, key_cols)
        n_keys += pending.num_rows
        if warm_keys:
            if warm_start_values is None:
                warm_start_values = pa.array(sorted(warm_starts))
            overlap = pending.filter(
                pc.is_in(pending.column("start"), value_set=warm_start_values)
            )
            for k in _keys(overlap, key_cols):
                if k in warm_keys:
                    n_dupes += 1

    assert not regressions, (
        f"{plugin} chr{chrom}: {regressions} start regressions within a tier "
        f"(first at row {first_regression}) -- the shard violates its sorted "
        f"contract and point lookups will silently miss rows"
    )
    if key_cols:
        assert n_dupes == 0, (
            f"{plugin} chr{chrom}: {n_dupes:,} duplicate probe keys -- "
            f"assume_unique=true is WRONG for this source"
        )
        print(f"  assume_unique verified exhaustively over {n_keys:,} keys")
    print(f"  \u2713 {plugin} chr{chrom}")

In [ ]:
def cleanup(plugin, keep_source=False):
    """Free the disk for the next plugin. The parquet cache is never touched."""
    d = INPUT_ROOT / plugin
    if keep_source or not d.exists():
        print(f"  keeping {d}")
        return
    size = sum(f.stat().st_size for f in d.rglob("*") if f.is_file()) / 2**30
    shutil.rmtree(d)
    print(f"  removed {d} ({size:.1f} GB), free now {free_gb():.1f} GB")


def scratch_check(plugin):
    """#217's ScratchGuard should leave no .tmp behind, even after a failed build."""
    leftovers = list((CACHE_ROOT / "plugin" / plugin).glob("*.tmp"))
    print("  scratch files:", leftovers or "none ✓")
    return leftovers

## 3. Smoke run — chr21 for every plugin

**Run this before any full-genome build.** chr21 is the smallest autosome, so the whole
matrix (5 plugins × download → chromosome-select → build → validate) completes quickly and proves the
pipeline end to end. A failure here costs minutes; the same failure found during CADD chr1
costs hours.

Each plugin's input is deleted before the next is downloaded — that ordering is what keeps
the run inside the disk budget.

### 3.1 clinvar

In [ ]:
download("clinvar")
build_chrom("clinvar", 21)
validate("clinvar", 21)
scratch_check("clinvar")
cleanup("clinvar")  # comment out to keep the source for the full-genome run below

### 3.2 alphamissense

In [ ]:
download("alphamissense")
build_chrom("alphamissense", 21)
validate("alphamissense", 21)
scratch_check("alphamissense")
cleanup("alphamissense")  # comment out to keep the source for the full-genome run below

### 3.3 spliceai

In [ ]:
download("spliceai")
build_chrom("spliceai", 21)
validate("spliceai", 21)
scratch_check("spliceai")
cleanup("spliceai")  # comment out to keep the source for the full-genome run below

### 3.4 dbnsfp

In [ ]:
download("dbnsfp")
build_chrom("dbnsfp", 21)
validate("dbnsfp", 21)
scratch_check("dbnsfp")
cleanup("dbnsfp")  # comment out to keep the source for the full-genome run below

### 3.5 cadd

In [ ]:
download("cadd")
build_chrom("cadd", 21)
validate("cadd", 21)
scratch_check("cadd")
cleanup("cadd")  # comment out to keep the source for the full-genome run below

### 3.6 CADD — does final DataFusion ordering hold on real multi-source data?

CADD is the real multi-source exercise for #217. Its canonical SNV and indel BGZF
files stay separate; Rust queries the current chromosome from both, registers the
results as named source parts, and combines them with `UNION ALL` in `ingest_sql`.

Nothing is pre-sorted, deliberately: the relevant `start` exists only after
allele normalization in SQL. The final DataFusion `ORDER BY tier, start` must make
the published shard monotonic regardless of source or join order; `validate()`
checks that storage contract exhaustively.

In [ ]:
def cadd_ordering_check(chrom=21):
    """Exercise final ordering on CADD's two real source parts.

    The input parts are not merged or sorted outside DataFusion. The build must
    succeed and the shard must satisfy the sorted storage contract, which
    validate() proves.
    """
    build_chrom("cadd", chrom)
    validate("cadd", chrom)
    print("\n[ok] multi-source input produced a correctly ordered shard")


# cadd_ordering_check(21)

## 4. Full genome, one plugin at a time

Only after §3 is green for all five. Re-downloads the plugin's source, walks chr1..22 + X,
and deletes the source at the end.

**Budget realistically.** CADD chr1 alone runs ~2-4.5 h; the full CADD genome is a
multi-day job. Run one plugin per session and check the shard after each chromosome —
`build_chrom` is idempotent per chromosome (`overwrite=True`), so an interrupted run
resumes by re-running the loop with the finished chromosomes removed from `CHROMS`.

In [ ]:
# Autosomes only: that is the scope the plugin caches, the VEP
# references, and the parity gate all cover. chrX has no validated
# reference here, so add it deliberately rather than by default.
CHROMS = [str(c) for c in range(1, 23)]


def build_genome(plugin, chroms=CHROMS, stop_on_error=True):
    download(plugin)
    done, failed = [], []
    for c in chroms:
        print(f"\n=== {plugin} chr{c} ({free_gb():.0f} GB free) ===")
        try:
            build_chrom(plugin, c)
            validate(plugin, c)
            done.append(c)
        except Exception as e:
            print(f"  FAILED {plugin} chr{c}: {e}")
            failed.append((c, str(e)))
            if stop_on_error:
                raise
    print(f"\n{plugin}: {len(done)} ok, {len(failed)} failed -> {failed}")
    scratch_check(plugin)
    cleanup(plugin)
    return done, failed

In [ ]:
# One per session. Order matters: smallest first, biggest last.
# build_genome("clinvar")
# build_genome("alphamissense")
# build_genome("spliceai")
# build_genome("dbnsfp")
# build_genome("cadd")

## 5. Result summary

What to report back on #217: per-plugin chromosome coverage, row counts, and — the point of
the exercise — whether anything the PR touches misbehaved on real data.

In [ ]:
rows = []
for plugin in PLUGINS:
    d = CACHE_ROOT / "plugin" / plugin
    if not d.is_dir():
        continue
    shards = sorted(d.glob("chr*.parquet"))
    total = sum(pq.read_metadata(s).num_rows for s in shards)
    size = sum(s.stat().st_size for s in shards) / 2**30
    rows.append((plugin, len(shards), total, size))
    print(f"{plugin:<14} {len(shards):>3} shards  {total:>15,} rows  {size:>6.1f} GB")
print(f"\ncache total: {sum(r[3] for r in rows):.1f} GB in {CACHE_ROOT}")